# AI Reply Assistant: train your personal reply model

This notebook fine-tunes a small open model (**Llama 3.2 3B Instruct**) to write replies **in your style**, then exports it for **Ollama** so the extension can use it.

**Two stages (LoRA fine-tuning):**
1. **General reply skill:** about 1,500 message → reply pairs from the public **SODA** dialogue dataset (allenai/soda, CC-BY-4.0). This teaches the model to answer *as the person who received the message*.
2. **Your style:** your own replies exported from the extension (*What I've learned → Settings & data → Export training data (JSONL)*).

**Evaluation:** 20% of your replies are held out. The notebook compares the base model, the model after stage 1, and after stage 2, using the same edit-distance metric as the extension's dashboard, plus ROUGE-L.

**How to run**
1. *Runtime → Change runtime type → T4 GPU* (free).
2. *Runtime → Run all*. When cell 3 asks, upload your `ai-reply-training-*.jsonl` file (or skip it; stage 2 is then skipped).
3. It takes roughly 30–45 minutes in total, most of it the GGUF export at the end.
4. Follow the last cell to load the model into Ollama on your laptop.

**Privacy:** your uploaded replies are processed on Google's Colab servers for this session.

## 1. Install (about 3 minutes)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q unsloth

## 2. Settings
Change these if you like. `OLLAMA_MODEL_NAME` is the name the model gets in Ollama and in the extension's model list.

In [ ]:
BASE_MODEL        = "unsloth/Llama-3.2-3B-Instruct"   # same family as the extension's llama3.2
YOUR_NAME         = "Aditya"                          # used in the system prompt
OLLAMA_MODEL_NAME = "aditya-reply"
STAGE1_EXAMPLES   = 1500      # public SODA pairs for stage 1 (0 = skip stage 1)
STAGE2_EPOCHS     = 3         # passes over your own replies (small data: 2-4)
HOLDOUT_SHARE     = 0.2       # share of your replies kept aside for evaluation
MAX_SEQ_LENGTH    = 1024
QUANTIZATION      = "q4_k_m"  # GGUF size/quality: q4_k_m (~2 GB) or q8_0 (~3.4 GB)
SEED              = 3407

import random
random.seed(SEED)

## 3. Upload your exported replies (JSONL)
Export them in the extension: **What I've learned → Settings & data → ⬇ Export training data (JSONL)**. Press *Cancel* to skip (stage 2 and the personal evaluation are then skipped).

In [ ]:
import json, io

personal_raw = []
try:
    from google.colab import files
    uploaded = files.upload()
    for name, data in uploaded.items():
        for line in io.StringIO(data.decode("utf-8")):
            line = line.strip()
            if line:
                personal_raw.append(json.loads(line))
except Exception as e:
    print("No file uploaded:", e)

def parse_personal(row):
    """Extension JSONL row -> (message, reply). The user turn starts with 'MESSAGE THE USER RECEIVED (site):'."""
    msgs = row.get("messages", [])
    user = next((m["content"] for m in msgs if m["role"] == "user"), "")
    reply = next((m["content"] for m in msgs if m["role"] == "assistant"), "")
    if user.startswith("MESSAGE THE USER RECEIVED"):
        user = user.split("\n", 1)[1] if "\n" in user else ""
    return user.strip(), reply.strip()

personal_pairs = [p for p in (parse_personal(r) for r in personal_raw) if p[0] and p[1]]
print(f"Loaded {len(personal_pairs)} of your replies.")
if 0 < len(personal_pairs) < 30:
    print("Note: fewer than 30 replies. Stage 2 will still run, but expect a modest effect; re-run later with more data.")

## 4. Prompt format (identical to the extension's)
Training uses the same wording the extension sends, so the trained model behaves the same way inside the extension.

In [ ]:
def system_prompt(name=YOUR_NAME):
    who = f" The user's name is {name}." if name else ""
    return ("You are a ghost-writer. You write the reply that the USER will send, in the user's own voice, "
            "to the sender of the message. Write in first person. Do not offer the user help or ask the user questions. "
            "Output ONLY the reply text, ready to send." + who)

def user_turn(message):
    return (f'MESSAGE THE USER RECEIVED:\n"""\n{message.strip()[:2500]}\n"""\n\n'
            "Now write the user's reply to the sender of this message. Output only the reply.")

def to_messages(message, reply=None, name=YOUR_NAME):
    m = [{"role": "system", "content": system_prompt(name)},
         {"role": "user", "content": user_turn(message)}]
    if reply is not None:
        m.append({"role": "assistant", "content": reply.strip()})
    return m

print(json.dumps(to_messages("Can you join the meeting at 10?", "Sure, see you at 10!"), indent=1))

## 5. Stage 1 data: public reply pairs (SODA)
Takes one message → reply pair from each of the first dialogues of SODA (streamed, so nothing large is downloaded).

In [ ]:
from datasets import load_dataset, Dataset

def soda_pairs(n):
    pairs = []
    if n <= 0:
        return pairs
    try:
        stream = load_dataset("allenai/soda", split="train", streaming=True)
        for row in stream:
            turns, speakers = row.get("dialogue") or [], row.get("speakers") or []
            options = [i for i in range(len(turns) - 1)
                       if i + 1 < len(speakers) and speakers[i] != speakers[i + 1]
                       and 20 <= len(turns[i]) <= 400 and 8 <= len(turns[i + 1]) <= 300]
            if options:
                i = random.choice(options)
                pairs.append((turns[i], turns[i + 1]))
            if len(pairs) >= n:
                break
    except Exception as e:
        print("Could not load SODA, stage 1 will be skipped:", e)
    return pairs

stage1_pairs = soda_pairs(STAGE1_EXAMPLES)
print(f"Stage 1: {len(stage1_pairs)} public pairs")
for m, r in stage1_pairs[:3]:
    print("\nMESSAGE:", m, "\nREPLY:  ", r)

## 6. Load the base model with LoRA adapters

In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template, train_on_responses_only
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL, max_seq_length=MAX_SEQ_LENGTH, dtype=None, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16, lora_dropout=0, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=SEED)
tokenizer = get_chat_template(tokenizer, chat_template="llama-3.1")
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

## 7. Helpers: training, generation, metrics

In [ ]:
import inspect, re
from trl import SFTTrainer, SFTConfig

def to_text_dataset(pairs, name):
    rows = [{"text": tokenizer.apply_chat_template(to_messages(m, r, name), tokenize=False)} for m, r in pairs]
    return Dataset.from_list(rows)

def make_trainer(dataset, epochs, lr, batch=2, accum=4):
    # Works across TRL versions (argument names changed over time)
    cfg = dict(per_device_train_batch_size=batch, gradient_accumulation_steps=accum, num_train_epochs=epochs,
               learning_rate=lr, warmup_steps=5, logging_steps=10, optim="adamw_8bit", weight_decay=0.01,
               lr_scheduler_type="linear", seed=SEED, output_dir="outputs", report_to="none",
               dataset_text_field="text", max_seq_length=MAX_SEQ_LENGTH)
    params = inspect.signature(SFTConfig.__init__).parameters
    if "max_seq_length" not in params and "max_length" in params:
        cfg["max_length"] = cfg.pop("max_seq_length")
    args = SFTConfig(**{k: v for k, v in cfg.items() if k in params})
    tparams = inspect.signature(SFTTrainer.__init__).parameters
    kw = {"processing_class": tokenizer} if "processing_class" in tparams else {"tokenizer": tokenizer}
    trainer = SFTTrainer(model=model, train_dataset=dataset, args=args, **kw)
    # learn only the reply, not the prompt
    return train_on_responses_only(trainer,
        instruction_part="<|start_header_id|>user<|end_header_id|>\n\n",
        response_part="<|start_header_id|>assistant<|end_header_id|>\n\n")

@torch.no_grad()
def generate(message, name=YOUR_NAME, max_new_tokens=200):
    FastLanguageModel.for_inference(model)
    enc = tokenizer.apply_chat_template(to_messages(message, None, name), add_generation_prompt=True, return_tensors="pt")
    ids = (enc["input_ids"] if hasattr(enc, "keys") else enc).to("cuda")
    out = model.generate(input_ids=ids, max_new_tokens=max_new_tokens, do_sample=False,
                         pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()

def words(t):
    return t.split()

def edit_ratio(a, b):
    """Same metric as the extension dashboard: word-level edit distance / longer length (0 = identical)."""
    x, y = [w.lower() for w in words(a)][:400], [w.lower() for w in words(b)][:400]
    if not x and not y: return 0.0
    if not x or not y: return 1.0
    prev = list(range(len(y) + 1))
    for i in range(1, len(x) + 1):
        cur = [i]
        for j in range(1, len(y) + 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (x[i - 1] != y[j - 1])))
        prev = cur
    return min(1.0, prev[-1] / max(len(x), len(y)))

def rouge_l(pred, ref):
    """ROUGE-L F1 on lower-cased words (word overlap in order)."""
    p = re.findall(r"\w+", pred.lower()); r = re.findall(r"\w+", ref.lower())
    if not p or not r: return 0.0
    L = [[0] * (len(r) + 1) for _ in range(len(p) + 1)]
    for i in range(len(p)):
        for j in range(len(r)):
            L[i + 1][j + 1] = L[i][j] + 1 if p[i] == r[j] else max(L[i][j + 1], L[i + 1][j])
    lcs = L[-1][-1]
    if lcs == 0: return 0.0
    prec, rec = lcs / len(p), lcs / len(r)
    return 2 * prec * rec / (prec + rec)

def evaluate(pairs, label, use_adapter=True):
    rows = []
    for m, ref in pairs:
        if use_adapter:
            pred = generate(m)
        else:
            with model.disable_adapter():
                pred = generate(m)
        rows.append({"edit": edit_ratio(pred, ref), "rougeL": rouge_l(pred, ref),
                     "len_diff": abs(len(words(pred)) - len(words(ref))), "pred": pred, "ref": ref, "msg": m})
    n = max(1, len(rows))
    summary = {"model": label, "examples": len(rows),
               "edit_amount": sum(r["edit"] for r in rows) / n,
               "rougeL": sum(r["rougeL"] for r in rows) / n,
               "length_diff_words": sum(r["len_diff"] for r in rows) / n}
    print(f"{label:<28} edit {summary['edit_amount']:.0%}   ROUGE-L {summary['rougeL']:.2f}   length diff {summary['length_diff_words']:.1f} words")
    return summary, rows

## 8. Split your replies: train vs. held-out test
The test replies are never trained on, so the comparison is fair. With no personal data, 30 held-out SODA pairs are used instead (measures general reply quality, not your style).

In [ ]:
pairs = personal_pairs[:]
random.shuffle(pairs)
n_test = max(3, round(len(pairs) * HOLDOUT_SHARE)) if len(pairs) >= 6 else 0
personal_test, personal_train = pairs[:n_test], pairs[n_test:]
if personal_test:
    test_pairs, test_kind = personal_test, "your held-out replies"
else:
    test_pairs, test_kind = stage1_pairs[-30:], "held-out SODA pairs"
    stage1_pairs = stage1_pairs[:-30]
print(f"Train on {len(personal_train)} of your replies; test on {len(test_pairs)} {test_kind}.")

## 9. Baseline: the base model before any training

In [ ]:
results = []
s, base_rows = evaluate(test_pairs, "Base Llama 3.2 3B", use_adapter=False)
results.append(s)

## 10. Stage 1: general reply skill (about 5–10 minutes on a T4)

In [ ]:
if stage1_pairs:
    FastLanguageModel.for_training(model)
    make_trainer(to_text_dataset(stage1_pairs, name=""), epochs=1, lr=2e-4).train()
    s, stage1_rows = evaluate(test_pairs, "After stage 1 (public)")
    results.append(s)
else:
    print("Stage 1 skipped.")

## 11. Stage 2: your personal style

In [ ]:
if personal_train:
    FastLanguageModel.for_training(model)
    make_trainer(to_text_dataset(personal_train, name=YOUR_NAME), epochs=STAGE2_EPOCHS, lr=1e-4, batch=2, accum=2).train()
    s, stage2_rows = evaluate(test_pairs, "After stage 2 (personal)")
    results.append(s)
else:
    print("Stage 2 skipped (no personal replies uploaded).")

## 12. Results
Lower **edit amount** and **length difference**, and higher **ROUGE-L**, mean replies closer to what you actually sent. With few test replies the numbers are noisy: report the number of examples next to them.

In [ ]:
import pandas as pd
df = pd.DataFrame(results)
df["edit_amount"] = df["edit_amount"].map(lambda v: f"{v:.0%}")
df["rougeL"] = df["rougeL"].map(lambda v: f"{v:.2f}")
df["length_diff_words"] = df["length_diff_words"].map(lambda v: f"{v:.1f}")
print(f"Test set: {len(test_pairs)} {test_kind}\n")
print(df.to_markdown(index=False))
df.to_csv("evaluation_results.csv", index=False)

# Side-by-side examples for the report
last_rows = globals().get("stage2_rows") or globals().get("stage1_rows") or base_rows
for b, a in list(zip(base_rows, last_rows))[:5]:
    print("\n" + "-" * 80)
    print("MESSAGE:     ", a["msg"][:300])
    print("YOU SENT:    ", a["ref"])
    print("BASE MODEL:  ", b["pred"])
    print("FINE-TUNED:  ", a["pred"])

## 13. Try it on your own message

In [ ]:
print(generate("Hi Aditya, are you free to review my slides before Friday's presentation?\nThanks,\nRiya"))

## 14. Export for Ollama (GGUF, about 10–15 minutes)
Merges the LoRA adapters into the model, converts it to GGUF, writes an Ollama `Modelfile`, and saves both to your Google Drive (folder `ai-reply-model`). Downloading a ~2 GB file from Drive is more reliable than a direct browser download.

In [ ]:
import glob, os, shutil

OUT_DIR = "reply_model_gguf"
model.save_pretrained_gguf(OUT_DIR, tokenizer, quantization_method=QUANTIZATION)

ggufs = sorted(glob.glob(f"{OUT_DIR}/**/*.gguf", recursive=True) + glob.glob("*.gguf"), key=os.path.getsize)
gguf = ggufs[-1]
print("GGUF file:", gguf, f"({os.path.getsize(gguf) / 1e9:.2f} GB)")

# Llama 3 chat template for Ollama, same format the model was trained on
TEMPLATE = ("{{ if .System }}<|start_header_id|>system<|end_header_id|>\n\n"
            "{{ .System }}<|eot_id|>{{ end }}{{ range .Messages }}<|start_header_id|>{{ .Role }}<|end_header_id|>\n\n"
            "{{ .Content }}<|eot_id|>{{ end }}<|start_header_id|>assistant<|end_header_id|>\n\n")
modelfile = ("FROM ./" + os.path.basename(gguf) + "\n"
             + 'TEMPLATE """' + TEMPLATE + '"""\n'
             + 'PARAMETER stop "<|eot_id|>"\n'
             + 'PARAMETER stop "<|start_header_id|>"\n'
             + "PARAMETER temperature 0.4\n")
with open("Modelfile", "w") as f:
    f.write(modelfile)
print(modelfile)

from google.colab import drive
drive.mount("/content/drive")
dest = "/content/drive/MyDrive/ai-reply-model"
os.makedirs(dest, exist_ok=True)
shutil.copy(gguf, dest)
shutil.copy("Modelfile", dest)
shutil.copy("evaluation_results.csv", dest)
print("Saved to Google Drive →", dest, os.listdir(dest))

## 15. Load it into Ollama on your laptop
1. From Google Drive, download the folder **ai-reply-model** (the `.gguf` file, `Modelfile`, `evaluation_results.csv`) into one folder on your laptop, e.g. `C:\ai-reply-model`.
2. Open PowerShell in that folder and run:
   ```
   cd C:\ai-reply-model
   ollama create aditya-reply -f Modelfile
   ollama run aditya-reply "Can you join the meeting at 10 tomorrow?"
   ```
3. In the extension: **Settings → Local LLM → 🔍 Detect Models**, select **aditya-reply**, and click **Save**.

To retrain later with more replies, export the JSONL again and re-run this notebook. Keep the extension's personalisation switched on: the fine-tuned model and the learned profile work together.